In [107]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

path = r"D:\pythonAi\study\DetecTion\docs"

loader = DirectoryLoader(
    path,
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    silent_errors=True   
)

docs = loader.load()

print(len(docs))

126


In [108]:
import shutil, os
if os.path.exists("faiss_index"):
    shutil.rmtree("faiss_index")

In [109]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
textSplitter = RecursiveCharacterTextSplitter(chunk_size=512,
                                              chunk_overlap=50)
documents = textSplitter.split_documents(docs)

In [110]:
documents

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-07-14T13:00:48+05:30', 'author': 'IEC', 'moddate': '2023-07-14T13:00:48+05:30', 'source': 'D:\\pythonAi\\study\\DetecTion\\docs\\10-Laboratory_Report.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content="GOVERNMENT OF MEGHALAYA \nHEALTH & FAMILY WELFARE DEPARTMENT \n \n \nLABORATORY/INVESTIGATION REPORT  \n \nPatient's Name:   MRD No./UHID No.   \nAge:   Gender M F O Ward No.   \nDate of Admission DD/MM/YY Bed No.   \nProvisional Diagnosis:   \n \nSL \nNo. Name of Investigation Date of Request Results/Findings Critical Values \n1 2 3 1 2 3 1 2 3 \n1 ABO+ Rh+-                   \n2 Haemoglobin gm%                   \n3 HbA1c                   \n4 Hbvario                   \n5 TLC                   \n6 DLC"),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-07-14T13:00:48+05:30', 'author': 'IEC', 'm

In [111]:
import re
def cleantext(text):
   if not text or not isinstance(text,str):
    return None
   text = text.encode("utf-8","ignore").decode("utf-8","ignore")
   text = re.sub(r'\s+',' ',text).strip()
   text = re.sub(r'page \d+', '',text)
   return text if text.strip() else None


cleanedtext = []
for i in documents:
    cleaned = cleantext(i.page_content)
    if cleaned:
      i.page_content=cleantext(i.page_content)
      cleanedtext.append(i)
    

In [112]:
from langchain_ollama import OllamaEmbeddings
emb = OllamaEmbeddings(model='mxbai-embed-large:latest')

In [113]:
from langchain_community.vectorstores import FAISS
# db = FAISS.from_documents(cleanedtext,emb)

In [114]:
import os
if os.path.exists("faiss_index"):
    db = FAISS.load_local("faiss_index", emb, allow_dangerous_deserialization=True)
else:
    db = FAISS.from_documents(cleanedtext, emb)
    db.save_local("faiss_index")

In [134]:
retriever = db.as_retriever(
    search_kwargs = {"k":3,"fetch_k":10,"lambda_mult":0.85},
    search_type = 'mmr'
)

In [135]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(
[
    ("system", """You are an expert medical assistant analyzing lab reports
     Use ONLY the provided context below to answer.
     Do NOT use outside knowleadge.
     If information is not found, say :"Not found in available reports.'
     Always flag abnormal values with ⚠️
     Never provide diagnosis - only report what data shows.
     
     context (Lab Reports):
     {context}"""),
     ("human","{question}")
]
)

In [136]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="phi3:mini",
                 temperature=0.2,
                 system="You are a medical assistant. Answer ONLY from the provided context. If not found, say 'Not found in availaible reports.")


In [137]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [138]:
# query1 = "Generate a summary of blood tests"
# response1 = rag_chain.invoke(query1)
# print(response1)

In [139]:
query2 = "what is hCG humanchorionic gonadotrophin"
response = rag_chain.invoke(query2)
print(response)

hCG, or Human Chorionic Gonadotropin, is a hormone secreted by the placenta and consists of α and β subunits. The presence of its β subunit in non-pregnant females and males can be indicative of cancer. It has prognostic importance as an increase or decrease may indicate disease stage or response to therapy, respectively.


In [140]:
docs_retrieved = retriever.invoke(" What is hcg humanchrionic gonadotrophin")
for i, doc in enumerate(docs_retrieved):
    print(f"\n---chunk {i+1}---")
    print(doc.page_content[:300])
    print(f"Source: {doc.metadata.get('source','unknown')}")


---chunk 1---
10 S/CTC/Training/Lab tests/May 2014 LABORATORY TESTS TUMOUR MARKERS continued (useful in staging disease as wells as monitoring response to treatment) TEST FUNCTION INCREASED hCG (human chorionic gonadotrophin) Hormone secreted by the placenta and consisting of α and β subunits. The β subunits is n
Source: D:\pythonAi\study\DetecTion\docs\LABORATORY TESTS v5 may 15_30052017_0.pdf

---chunk 2---
ICSH (interstitial cell- stimulating hormone) In females this hormone triggers ovulation and in men produces testosterone mIU/mL  Post-menopause  Polycystic ovarian syndrome  Pituitary failure Oestradiol Sex hormone found in both sexes. Regulates female organ and tissue development. In males it r
Source: D:\pythonAi\study\DetecTion\docs\LABORATORY TESTS v5 may 15_30052017_0.pdf

---chunk 3---
6 S/CTC/Training/Lab tests/May 2014 LABORATORY TESTS GONAD FUNCTION TESTS TEST FUNCTION UNITS INCREASED DECREASED FSH Follicle Stimulating Hormone Hormone in the endocrine system. FSH reg